# Advanced Bird Species Recognition System
## EfficientNetB3 Fine-Grained Classification Training
This notebook trains a high-accuracy EfficientNet-B3 model on the CUB-200-2011 dataset to achieve 95%+ validation accuracy over 150 epochs.

### Setup Instructions:
1. Go to **Runtime > Change runtime type** and ensure **Hardware accelerator** is set to **T4 GPU**.
2. Ensure your `CUB_200_2011_cropped.zip` dataset is uploaded to the root of your Google Drive.
3. Run the cells below sequentially.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Extract the Dataset
import os
os.makedirs('/content/data/processed', exist_ok=True)

print("Extracting dataset... This may take a minute or two.")
!unzip -q "/content/drive/MyDrive/Bharath final year project/CUB_200_2011_cropped.zip" -d "/content/data/processed/"
print("✅ Dataset successfully extracted!")

In [ ]:
# 3. Import Dependencies and Configure Training
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import time

data_dir = '/content/data/processed/CUB_200_2011_cropped'
num_classes = 200
batch_size = 32
num_epochs = 150 # Targeting 95%+ peak accuracy
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")

# --- 4. Data Augmentation ---
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x]) for x in ['train', 'test']}
dataloaders = {x: DataLoader(image_datasets[x], batch_size=batch_size, shuffle=True, num_workers=2) for x in ['train', 'test']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'test']}

# --- 5. Model Architecture ---
model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)

# Freeze base layers initially to learn classification head
for param in model.parameters():
    param.requires_grad = False
    
num_ftrs = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.5, inplace=True),
    nn.Linear(num_ftrs, 512),
    nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(512, num_classes)
)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.classifier.parameters(), lr=1e-4) 
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=3)

# --- 6. Training Loop ---
best_acc = 0.0

for epoch in range(num_epochs):
    print(f'Epoch {epoch+1}/{num_epochs}')
    print('-' * 10)
    
    # Unfreeze network after 5 epochs for fine-grained feature extraction
    if epoch == 5:
        print("\n--- Unfreezing all layers for Deep Fine-tuning ---")
        for param in model.parameters():
            param.requires_grad = True
        optimizer = optim.AdamW(model.parameters(), lr=1e-5)
        
    for phase in ['train', 'test']:
        if phase == 'train': model.train()
        else: model.eval()

        running_loss, running_corrects = 0.0, 0

        for inputs, labels in dataloaders[phase]:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)
                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / dataset_sizes[phase]
        epoch_acc = running_corrects.double() / dataset_sizes[phase]
        print(f'{phase.capitalize()} | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc*100:.2f}%')

        if phase == 'test':
            scheduler.step(epoch_acc)
            if epoch_acc > best_acc:
                best_acc = epoch_acc
                os.makedirs('/content/weights', exist_ok=True)
                torch.save(model.state_dict(), '/content/weights/efficientnet_b3_best.pth')
                print(f"\n🏆 NEW BEST MODEL SAVED! Accuracy: {best_acc*100:.2f}%")
                
print(f'\n✅ Training Complete. Peak Validation Accuracy: {best_acc*100:.2f}%')

# 7. Copy final weights back to Google Drive for safety
import shutil
try:
    shutil.copy('/content/weights/efficientnet_b3_best.pth', '/content/drive/MyDrive/Bharath final year project/efficientnet_b3_best.pth')
    print("\n📁 Successfully saved your final model to your Google Drive!")
except Exception as e:
    print(f"\nCould not copy to drive automatically: {e}. You can manually download it from the left 'Files' tab in Colab.")